# 02 — CNN Object Detection Training with Ray Data + Ray Train + Lance

**Purpose:** Train a CNN object detection model on BDD100K's 10-category bounding box labels using the Lance dataset built in `01_create_lance_dataset.ipynb`. Demonstrates why `ray.data.read_lance` + Ray Train is the right stack for distributed image training on Databricks — and benchmarks it against the Delta/Parquet equivalent.

---

## Why Ray Data + Ray Train instead of Spark + Delta?

The standard Databricks path for model training routes data through Spark: read Delta table → convert to Pandas/NumPy → feed PyTorch DataLoader. For tabular features this works. For raw image training it breaks down at scale for three reasons:

**1. Parquet deserialization burns CPU on every batch.** Ray Data's in-memory engine represents data as Apache Arrow blocks. Reading from Delta/Parquet requires: storage read → Parquet deserialization → Arrow conversion. Lance is Arrow-native — Ray Data reads directly into Arrow blocks with no deserialization step. That CPU budget goes to image augmentation instead.

**2. Row-group random access is expensive for binary data.** At ~100KB/frame, Parquet row groups shrink to ~1,280 rows before hitting the 128MB limit. Every random batch read loads a full 128MB row group to retrieve 1,280 samples — even if your batch size is 64. Lance's fragment-level addressing fetches exactly the rows requested, O(1) per row regardless of dataset size.

**3. Fragment-parallel reads align to Ray's actor model.** A Lance dataset is composed of independent fragments. Each Ray worker actor reads its assigned fragment(s) with no cross-worker coordination. Object storage pressure is predictable and coalesced — no thundering herd of per-image GET requests.

---

## What this notebook does

1. **Load the Lance dataset.** Open the Lance `frames` dataset from `01_create_lance_dataset.ipynb`. Pin the dataset version for training reproducibility — any future appends or schema updates won't affect this run.

2. **Build the Ray Data pipeline.** Use `ray.data.read_lance` with column projection (`image`, `bbox_labels` only — metadata columns are not loaded during training). Define preprocessing transforms as Ray Data map operations: JPEG decode → resize to model input resolution → normalize → random horizontal flip. Preprocessing runs on CPU actors in parallel with GPU training, keeping the GPU saturated.

3. **Data loading benchmark.** Before training, run a controlled throughput comparison:
   - `ray.data.read_lance` on the Lance dataset
   - `ray.data.read_parquet` on an equivalent Delta table at the same row count
   Report samples/second for both. The gap widens past 10M rows where Delta's row-group overhead compounds per batch.

4. **Define the detection model.** ResNet-50 backbone with a Feature Pyramid Network (FPN) head, fine-tuned for BDD100K's 10 object categories: car, truck, bus, person, rider, bicycle, motorcycle, traffic light, traffic sign, train. Loss: classification cross-entropy + bounding box regression (smooth L1).

5. **Distributed training with Ray Train.** Configure a `TorchTrainer` with DDP (DistributedDataParallel) across all GPU workers in the Ray cluster. Each worker receives its shard of the Lance dataset via Ray Data — no manual data partitioning. Checkpoint model weights to UC Volumes at configurable intervals.

6. **Log to MLflow.** Track loss, mAP (mean Average Precision) per category, data loading throughput (samples/sec), and GPU utilization per epoch. Pin the Lance dataset version and frame sampling rate as run parameters so training runs are fully reproducible.

---

**Inputs:** Lance `frames` dataset at `/Volumes/{catalog}/{schema}/{volume}/frames/` (inline JPEG bytes + BDD100K bounding box labels)

**Outputs:** Trained ResNet-50 + FPN detection model logged to MLflow; Lance vs Delta throughput benchmark results